# Chapter 4 - Connectedness and Compactness

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 4, printed pp. 85-126; PDF pp. 103-144.

## Chapter Goal

This chapter turns two global topological questions into working tests for manifolds and spaces built from them. The connectedness question asks whether a space can be split by an open-and-closed yes/no decision. The compactness question asks whether every open cover has a finite witness. Those two ideas look abstract at first, but for manifolds they become practical: charts give small connected and precompact neighborhoods, path connectedness becomes equivalent to connectedness, and compactness powers closed-map and quotient arguments.

The notebook builds computational stand-ins for those tests. Finite graphs model component partitions and locally constant maps to a two-point discrete space. Sampled paths and the topologist's sine curve separate path connectedness from connectedness. Interval covers model finite subcover extraction and show why open intervals behave differently from closed intervals. Half-ball diagrams show why manifolds with boundary are locally compact and locally path-connected. The final lab asks you to classify several spaces by the invariants that the visuals expose.

The examples here are original teaching models. They are not proofs by themselves, but they are designed to expose the proof moves used throughout the chapter: clopen obstruction, continuous-image preservation, maximal component partition, finite-subcover selection, closed-subset inheritance, compact-neighborhood bases, and properness as compact preimage control.


## Computational Translation Guide

| Topological idea | Computational representation in this notebook | What to inspect |
| --- | --- | --- |
| Connectedness | A finite graph whose connected components control all locally constant binary labels | A connected probe only admits globally constant labels; a disconnected probe admits one label choice per component. |
| Clopen separation | A two-color decision that is constant on every edge or local path | Nonconstant decisions are exactly the finite analogue of a disconnection. |
| Path connectedness | Explicit line or curve witnesses, plus a sampled topologist's sine curve | Paths are easy to certify in convex or punctured Euclidean sets, but sampled closure behavior warns that connected need not imply path-connected. |
| Components and path components | NetworkX connected components and a property matrix | In locally path-connected spaces, components and path components coincide; outside that setting they may differ. |
| Compactness | Interval covers and finite subcover checks, plus closed-and-bounded Euclidean tests | Compact spaces convert infinite cover data into finite certificates. |
| Local compactness | Nested precompact neighborhoods in balls and half-balls | Manifold charts provide neighborhoods whose closures are compact subsets of Euclidean balls or half-balls. |
| Manifolds with boundary | The half-plane model $H^n$ and regular coordinate half-balls | Boundary points still have compact local closures, but their local chart shape is a half-ball rather than a full ball. |
| Proper maps | Preimages of compact target sets and sequences going to infinity | A proper map cannot send an escaping sequence into a compact part of the target. |

## Library Routing

- `networkx` is used for finite component probes and proof-dependency scaffolds. A graph is not a topology, but its component logic is the smallest executable model for clopen binary decisions.
- `matplotlib` is used for durable two-dimensional teaching diagrams: clopen cuts, sine-curve closure behavior, and half-ball neighborhoods. These are static invariants, so PNG is the right artifact.
- `plotly` is used for compactness and properness because hovering over covers, target bands, and preimage behavior makes finite-subcover and proper-map tests easier to inspect.
- `numpy` supplies sampled curves and numerical interval checks.
- `pandas` stores the applied lab's property matrix as an inspectable table artifact.

## Visual Storyboard

1. **Clopen cuts and components:** show a connected finite probe, a disconnected probe, and the binary-label invariant behind connectedness and maps to discrete spaces.
2. **Proof dependency map:** connect continuous images, interval connectedness, compact images, closed maps, local compactness, and proper maps.
3. **Path connectedness versus connectedness:** display a path-connected convex disk beside the topologist's sine curve to isolate the failure of path witnesses.
4. **Compactness as finite evidence:** compare a finite subcover of a closed interval with finite initial segments of an open-interval cover that always miss points near 0.
5. **Local compactness at a boundary point:** draw full-ball and half-ball chart neighborhoods with compact closures inside larger coordinate neighborhoods.
6. **Proper map preimage test:** compare a proper polynomial-like map with a nonproper oscillatory map whose compact target fiber has unbounded preimage.
7. **Applied lab table:** classify example spaces by connected, path-connected, compact, locally connected, locally compact, and boundary-model behavior.


In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Wedge
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from IPython.display import display


def discover_book_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, cwd / "Introduction-to-Topological-Manifolds"]
    for candidate in candidates:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "source_map.json").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not find Introduction-to-Topological-Manifolds root")


BOOK_ROOT = discover_book_root()
os.chdir(BOOK_ROOT)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    chapter_artifact_root,
    save_json,
    save_csv,
    save_matplotlib,
    save_plotly_html,
    assert_artifacts,
    display_artifact,
)
from utils.validation import image_stats

UNIT_KEY = "chapter-04-connectedness-and-compactness"
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"
TABLES = ARTIFACT_ROOT / "tables"

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 170, "font.size": 10})


def book_local(path):
    return Path(path).resolve().relative_to(BOOK_ROOT.resolve())


print("BOOK_ROOT discovered")
print(f"Artifact root = {book_local(ARTIFACT_ROOT).as_posix()}")


In [ ]:
visual_storyboard = [
    {
        "concept": "connectedness as absence of nontrivial clopen decisions",
        "representation": "finite component probe plus two-color locally constant labels",
        "library": "networkx + matplotlib",
        "artifact": "figures/connectedness-clopen-and-components.png",
        "inspection_target": "Connected probes have one component and only globally constant binary labels.",
        "validation": "binary locally constant label count equals 2**component_count",
    },
    {
        "concept": "proof moves linking connectedness and compactness results",
        "representation": "directed dependency graph",
        "library": "networkx + matplotlib",
        "artifact": "figures/proof-dependency-map.png",
        "inspection_target": "Continuous-image theorems feed IVT, compact-image, quotient, and proper-map arguments.",
        "validation": "directed graph is acyclic and contains required theorem nodes",
    },
    {
        "concept": "path connectedness is stronger than connectedness",
        "representation": "convex path witness contrasted with topologist sine curve closure",
        "library": "numpy + matplotlib",
        "artifact": "figures/path-connectedness-sine-curve.png",
        "inspection_target": "The sine curve accumulates on the vertical segment without visible path access to it.",
        "validation": "sampled oscillation count and bounded plotting window recorded",
    },
    {
        "concept": "compactness as finite subcover evidence",
        "representation": "interactive interval-cover comparison",
        "library": "plotly",
        "artifact": "html/compactness-open-covers.html",
        "inspection_target": "Finite cover certifies [0,1], while finite initial segments of (1/n,1) miss points near 0.",
        "validation": "exact interval-merging check and explicit missed points",
    },
    {
        "concept": "local compactness for manifolds with boundary",
        "representation": "regular coordinate balls and half-balls with compact closures",
        "library": "matplotlib",
        "artifact": "figures/local-compactness-half-balls.png",
        "inspection_target": "Interior and boundary chart neighborhoods both contain smaller precompact neighborhoods.",
        "validation": "r_small < r_large and closed half-ball is closed and bounded in the Euclidean chart",
    },
    {
        "concept": "proper maps preserve escape to infinity",
        "representation": "preimage bands for x^2 and sin(x)",
        "library": "plotly + numpy",
        "artifact": "html/proper-map-preimage-test.html",
        "inspection_target": "A compact target interval has bounded preimage under x^2; the fiber sin(x)=0 is unbounded.",
        "validation": "polynomial preimage bound finite; sampled sine zeros grow unbounded",
    },
    {
        "concept": "space classification lab",
        "representation": "property matrix for chapter examples",
        "library": "pandas",
        "artifact": "tables/space-property-lab.csv",
        "inspection_target": "Compare connectedness, path connectedness, compactness, local compactness, and boundary models.",
        "validation": "chapter implication checks hold for listed cards",
    },
]

storyboard_path = save_json(visual_storyboard, CHECKS / "visual-storyboard.json")
display(pd.DataFrame(visual_storyboard)[["concept", "representation", "library", "artifact"]])
display_artifact(book_local(storyboard_path))


## 1. Connectedness: No Nontrivial Yes/No Split

The chapter begins with a definition that is deliberately negative: a space is connected when it cannot be written as two disjoint nonempty open pieces. The more useful working version is the clopen test: a connected space has no subsets that are both open and closed except the empty set and the whole space. In practice, this means that a continuous map from a connected space into a discrete set cannot make a nonconstant decision. If the target is `{0,1}`, the preimage of `1` would be clopen.

The finite graph below is not meant to replace the topological definition. It is a probe for the same logic. Think of an edge as a local path that forbids a jump in a locally constant binary label. On a connected graph, the label must propagate through every edge until the whole graph has the same value. On a graph with two components, one component can receive label `0` and the other label `1`. This is the finite analogue of a disconnection.

The proof-dependency map records the main chapter pattern: connectedness and compactness are both preserved by continuous images, and those image theorems feed the intermediate value theorem, quotient arguments, closed-map lemma, proper maps, and manifold consequences.


In [ ]:
def locally_constant_binary_label_count(graph):
    components = list(nx.connected_components(graph))
    return 2 ** len(components), sorted(len(component) for component in components)


connected_probe = nx.path_graph(7)
disconnected_probe = nx.disjoint_union(nx.path_graph(4), nx.path_graph(3))
connected_count, connected_sizes = locally_constant_binary_label_count(connected_probe)
disconnected_count, disconnected_sizes = locally_constant_binary_label_count(disconnected_probe)

component_checks = {
    "connected_probe": {
        "component_count": nx.number_connected_components(connected_probe),
        "component_sizes": connected_sizes,
        "locally_constant_binary_label_count": connected_count,
    },
    "disconnected_probe": {
        "component_count": nx.number_connected_components(disconnected_probe),
        "component_sizes": disconnected_sizes,
        "locally_constant_binary_label_count": disconnected_count,
    },
    "invariant": "locally_constant_binary_label_count == 2**component_count",
}
component_check_path = save_json(component_checks, CHECKS / "connectedness-finite-probes.json")

fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.2), constrained_layout=True)
ax = axes[0]
ax.axhline(0, color="#222222", lw=1.2)
ax.plot([-2.5, -0.08], [0, 0], color="#2a6fbb", lw=8, solid_capstyle="round", label="x < 0")
ax.plot([0.08, 2.5], [0, 0], color="#c64f4f", lw=8, solid_capstyle="round", label="x > 0")
ax.scatter([0], [0], s=95, facecolors="white", edgecolors="#222222", linewidths=1.5, zorder=3)
ax.text(0, 0.22, "removed point", ha="center")
ax.set_title("A clopen split in R without 0")
ax.set_xlim(-2.7, 2.7)
ax.set_ylim(-0.5, 0.55)
ax.axis("off")
ax.legend(loc="lower center", ncol=2, frameon=False)

ax = axes[1]
pos = {i: (i, 0) for i in connected_probe.nodes}
nx.draw_networkx_edges(connected_probe, pos, ax=ax, width=2.6, edge_color="#555555")
nx.draw_networkx_nodes(connected_probe, pos, ax=ax, node_size=260, node_color="#4c9f70", edgecolors="white")
ax.text(3, 0.38, f"components = 1\nbinary labels = {connected_count}", ha="center")
ax.set_title("Connected finite probe")
ax.set_axis_off()

ax = axes[2]
pos = {node: (node, 0) for node in range(4)}
pos.update({node: (node - 3.1, -0.9) for node in range(4, 7)})
colors = ["#2a6fbb" if node < 4 else "#c64f4f" for node in disconnected_probe.nodes]
nx.draw_networkx_edges(disconnected_probe, pos, ax=ax, width=2.6, edge_color="#555555")
nx.draw_networkx_nodes(disconnected_probe, pos, ax=ax, node_size=260, node_color=colors, edgecolors="white")
ax.text(1.5, 0.38, "label 0", ha="center", color="#2a6fbb")
ax.text(2.0, -1.32, "label 1", ha="center", color="#c64f4f")
ax.text(4.2, -0.35, f"components = 2\nbinary labels = {disconnected_count}", ha="center")
ax.set_title("Disconnected finite probe")
ax.set_axis_off()
connectedness_png = save_matplotlib(fig, FIGURES / "connectedness-clopen-and-components.png")
plt.close(fig)

proof = nx.DiGraph()
proof_edges = [
    ("no nontrivial clopen", "connected"),
    ("connected", "continuous image connected"),
    ("intervals in R", "intermediate value theorem"),
    ("continuous image connected", "intermediate value theorem"),
    ("paths from a basepoint", "path connected => connected"),
    ("locally path connected", "components = path components"),
    ("components = path components", "manifold connected iff path connected"),
    ("compact", "continuous image compact"),
    ("compact + Hausdorff", "closed map lemma"),
    ("continuous image compact", "closed map lemma"),
    ("regular coordinate balls", "manifolds locally compact"),
    ("regular half-balls", "boundary manifolds locally compact"),
    ("local compact Hausdorff", "paracompactness theorem"),
    ("closed map lemma", "quotient recognition"),
    ("proper map", "proper maps are closed"),
    ("compactly generated Hausdorff target", "proper maps are closed"),
]
proof.add_edges_from(proof_edges)
proof_checks = {
    "node_count": proof.number_of_nodes(),
    "edge_count": proof.number_of_edges(),
    "is_dag": nx.is_directed_acyclic_graph(proof),
    "required_nodes_present": all(
        node in proof.nodes
        for node in ["connected", "compact", "closed map lemma", "manifolds locally compact", "proper maps are closed"]
    ),
}
proof_check_path = save_json(proof_checks, CHECKS / "proof-dependency-checks.json")

fig, ax = plt.subplots(figsize=(11.2, 6.4), constrained_layout=True)
levels = {
    "no nontrivial clopen": (0, 3),
    "connected": (1, 3),
    "continuous image connected": (2, 3),
    "intervals in R": (2, 4),
    "intermediate value theorem": (3, 3.6),
    "paths from a basepoint": (0, 2),
    "path connected => connected": (1, 2),
    "locally path connected": (0, 1),
    "components = path components": (1, 1),
    "manifold connected iff path connected": (2.25, 1),
    "compact": (0, -0.5),
    "continuous image compact": (1.15, -0.5),
    "compact + Hausdorff": (1.15, -1.35),
    "closed map lemma": (2.4, -0.85),
    "quotient recognition": (3.65, -0.85),
    "regular coordinate balls": (0, -2.35),
    "regular half-balls": (0, -3.1),
    "manifolds locally compact": (1.35, -2.35),
    "boundary manifolds locally compact": (1.35, -3.1),
    "local compact Hausdorff": (2.55, -2.7),
    "paracompactness theorem": (3.75, -2.7),
    "proper map": (2.55, -1.75),
    "compactly generated Hausdorff target": (3.75, -1.75),
    "proper maps are closed": (4.75, -1.75),
}
node_colors = ["#d9b44a" if ("compact" in node or "proper" in node or "closed map" in node) else "#6aa6a6" if ("manifold" in node or "half" in node or "regular" in node) else "#8aa6d9" for node in proof.nodes]
nx.draw_networkx_edges(proof, levels, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=13, width=1.15, edge_color="#777777")
nx.draw_networkx_nodes(proof, levels, ax=ax, node_size=1550, node_color=node_colors, edgecolors="white", linewidths=1.0)
nx.draw_networkx_labels(proof, levels, ax=ax, font_size=7.2)
ax.set_title("Proof dependency scaffold for Chapter 4")
ax.set_axis_off()
proof_png = save_matplotlib(fig, FIGURES / "proof-dependency-map.png")
plt.close(fig)

display_artifact(book_local(connectedness_png), width=920)
display_artifact(book_local(proof_png), width=920)
display_artifact(book_local(component_check_path))
display_artifact(book_local(proof_check_path))
component_checks, proof_checks


## 2. Path Connectedness, Components, and Local Versions

A path is a continuous map from the unit interval into the space. Path connectedness is therefore constructive: to show two points are in the same path component, provide a path. Convex subsets of Euclidean space are the easiest example because the straight segment between two points stays inside the set. Punctured Euclidean space in dimension at least two is also path-connected because a path can detour around the missing point.

Connectedness is weaker. The topologist's sine curve is the chapter's key warning example. The oscillating graph approaches every point of the vertical segment at `x = 0`; adding that vertical segment gives a connected space because it is the closure of a connected graph piece. But a path cannot start on the vertical segment and then move into the oscillating graph without violating continuity. The visual below is a numerical picture of that obstruction: the graph accumulates everywhere along the segment, but the oscillations become infinitely compressed near the boundary line.

Local connectedness and local path connectedness repair this gap for manifolds. Euclidean balls and half-balls are path-connected, so manifolds have local bases by path-connected sets. In such spaces, path components are open and agree with components. That is why, inside the manifold category, authors often prove connectedness by drawing or constructing paths.


In [ ]:
t = np.linspace(0, 1, 300)
p = np.array([-0.75, -0.35])
q = np.array([0.65, 0.45])
segment = (1 - t)[:, None] * p + t[:, None] * q
x = np.linspace(0.012, 2 / np.pi, 2600)
y = np.sin(1 / x)
y_vertical = np.linspace(-1, 1, 500)
zero_crossings = int(np.sum(np.diff(np.signbit(y)) != 0))

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.2), constrained_layout=True)
ax = axes[0]
ax.add_patch(Circle((0, 0), 1.0, facecolor="#e9f3ee", edgecolor="#4c9f70", lw=2))
ax.plot(segment[:, 0], segment[:, 1], color="#1f6f43", lw=3)
ax.scatter([p[0], q[0]], [p[1], q[1]], color="#1f6f43", s=55, zorder=4)
ax.text(p[0] - 0.07, p[1] - 0.12, "p")
ax.text(q[0] + 0.05, q[1] + 0.05, "q")
ax.set_title("Convex set: straight path witness")
ax.set_aspect("equal")
ax.set_xlim(-1.12, 1.12)
ax.set_ylim(-1.12, 1.12)
ax.set_xticks([])
ax.set_yticks([])

ax = axes[1]
ax.plot(x, y, color="#2a6fbb", lw=1.15, label="y = sin(1/x), x > 0")
ax.plot(np.zeros_like(y_vertical), y_vertical, color="#c64f4f", lw=4, alpha=0.85, label="limit segment")
ax.axhline(0, color="#888888", lw=0.8)
ax.set_title("Connected closure, but no path access to the segment")
ax.set_xlim(-0.035, 0.68)
ax.set_ylim(-1.18, 1.18)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="upper right", frameon=False)
ax.text(0.16, -1.08, f"sampled zero crossings: {zero_crossings}", fontsize=9)
path_png = save_matplotlib(fig, FIGURES / "path-connectedness-sine-curve.png")
plt.close(fig)

path_checks = {
    "convex_segment_endpoint_error": float(np.linalg.norm(segment[0] - p) + np.linalg.norm(segment[-1] - q)),
    "convex_segment_max_radius": float(np.max(np.linalg.norm(segment, axis=1))),
    "sine_curve_sample_count": int(len(x)),
    "sine_curve_zero_crossings_in_window": zero_crossings,
    "sine_curve_x_min": float(x.min()),
    "sine_curve_x_max": float(x.max()),
    "topological_note": "The plotted oscillation is only a sample; the theorem uses closure of a connected set and a separate argument against path connectedness.",
}
path_check_path = save_json(path_checks, CHECKS / "path-connectedness-checks.json")
display_artifact(book_local(path_png), width=880)
display_artifact(book_local(path_check_path))
path_checks


## 3. Compactness: Finite Evidence from Infinite Data

Compactness says that an open cover always has a finite subcover. The definition is designed so continuous images behave well and so closed bounded Euclidean sets satisfy the same familiar theorems as closed intervals: extreme values exist, sequences have convergent subsequences in metric settings, and closed subsets stay compact.

The interval experiment below isolates the quantifier. The top row gives a finite open cover of the closed interval `[0,1]`; an exact interval-merging check verifies that the union contains every point of the closed interval. The bottom rows show the cover of `(0,1)` by intervals `(1/n,1)` for `n >= 2`. The whole infinite family covers `(0,1)`, but any finite initial segment has a smallest left endpoint and therefore misses a positive point closer to `0`. This is a compactness failure, not a plotting failure.

For manifolds, compactness has two roles. First, it gives global finite certificates from local chart data, such as finite coordinate covers of compact manifolds. Second, compactness interacts with Hausdorffness: compact subsets of Hausdorff spaces are closed, and a continuous map from a compact space to a Hausdorff space is closed. That closed-map lemma is one of the chapter's main engines for recognizing quotient maps, embeddings, and homeomorphisms.


In [ ]:
def merge_intervals(intervals):
    ordered = sorted(intervals)
    merged = []
    for left, right in ordered:
        if not merged or left > merged[-1][1]:
            merged.append([left, right])
        else:
            merged[-1][1] = max(merged[-1][1], right)
    return [(float(a), float(b)) for a, b in merged]


def open_intervals_cover_closed(intervals, a=0.0, b=1.0, eps=1e-12):
    merged = merge_intervals(intervals)
    if not merged or not (merged[0][0] < a + eps):
        return False
    right_edge = merged[0][1]
    for left, right in merged[1:]:
        if left > right_edge + eps:
            return False
        right_edge = max(right_edge, right)
    return right_edge > b - eps


closed_interval_cover = [(-0.12, 0.43), (0.35, 0.74), (0.68, 1.12)]
finite_tests = []
for N in range(2, 10):
    intervals = [(1 / n, 1.0) for n in range(2, N + 1)]
    missed_point = 1 / (N + 1)
    finite_tests.append({
        "N": N,
        "finite_union_left_endpoint": 1 / N,
        "missed_point": missed_point,
        "missed_point_is_in_open_interval": 0 < missed_point < 1,
        "covered_by_first_N": any(left < missed_point < right for left, right in intervals),
    })
cover_checks = {
    "closed_interval_cover": {
        "intervals": closed_interval_cover,
        "merged_intervals": merge_intervals(closed_interval_cover),
        "covers_closed_0_1": open_intervals_cover_closed(closed_interval_cover),
    },
    "open_interval_cover_family": "U_n = (1/n, 1) for n >= 2 covers (0,1), but no finite initial segment covers it.",
    "finite_initial_segment_tests": finite_tests,
}
cover_check_path = save_json(cover_checks, CHECKS / "compactness-cover-checks.json")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.18,
    subplot_titles=("Finite subcover for [0,1]", "Finite initial segments of U_n = (1/n,1) miss points near 0"),
)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 0], mode="lines", line=dict(color="black", width=5), name="[0,1]"), row=1, col=1)
for i, ((left, right), color) in enumerate(zip(closed_interval_cover, ["#2a6fbb", "#4c9f70", "#c64f4f"]), start=1):
    fig.add_trace(
        go.Scatter(x=[left, right], y=[i, i], mode="lines+markers", line=dict(color=color, width=10), marker=dict(size=6), name=f"closed-cover U{i}", hovertemplate=f"U{i}=({left:.2f},{right:.2f})<extra></extra>"),
        row=1,
        col=1,
    )
for N in range(2, 10):
    fig.add_trace(go.Scatter(x=[1 / N, 1], y=[N, N], mode="lines", line=dict(color="#6a6a6a", width=7), hovertemplate=f"finite union = (1/{N}, 1); misses 1/{N+1}<extra></extra>", showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=[1 / (N + 1)], y=[N], mode="markers", marker=dict(color="#c64f4f", size=9, symbol="x"), name="missed point" if N == 2 else None, showlegend=(N == 2)), row=2, col=1)
fig.update_xaxes(range=[-0.15, 1.15], title_text="coordinate in R", row=2, col=1)
fig.update_yaxes(visible=False, row=1, col=1)
fig.update_yaxes(visible=False, row=2, col=1)
fig.update_layout(height=610, width=900, template="plotly_white", legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0), margin=dict(l=40, r=30, t=95, b=45))
compact_html = save_plotly_html(fig, HTML / "compactness-open-covers.html")

display_artifact(book_local(compact_html), width=900, height=640)
display_artifact(book_local(cover_check_path))
cover_checks


## 4. Local Compactness and Manifolds with Boundary

A space is locally compact if each point has some compact neighborhood container. In Hausdorff spaces this becomes the more useful statement that each point has a basis of precompact open neighborhoods, meaning open sets whose closures are compact. For manifolds, charts supply this basis. A small Euclidean ball has compact closed ball closure; a boundary chart supplies the same idea with a half-ball.

The diagram shows the local model. An interior point has a full coordinate ball. A boundary point has a half-ball inside the closed half-plane model `H^2`. In both cases, a smaller open neighborhood has closure contained in a larger coordinate neighborhood, and the closure is a closed bounded subset of Euclidean space. That is exactly the reason manifolds with or without boundary are locally compact.

The proper-map visual applies compactness to maps. A map is proper when compact subsets of the target have compact preimages. For `x -> x^2`, the compact target interval `[0,4]` has preimage `[-2,2]`, still compact. For `x -> sin(x)` as a map into `R`, the compact set `{0}` has an unbounded preimage at integer multiples of `pi`, so the map is not proper. This sequence viewpoint is especially useful in first countable Hausdorff spaces: a proper map sends sequences escaping every compact set to sequences that also escape every compact set.


In [ ]:
r_small = 0.58
r_large = 0.92
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.3), constrained_layout=True)
ax = axes[0]
ax.add_patch(Circle((0, 0), r_large, facecolor="#edf3fb", edgecolor="#2a6fbb", lw=2, alpha=0.75))
ax.add_patch(Circle((0, 0), r_small, facecolor="#cfe8d8", edgecolor="#4c9f70", lw=2, alpha=0.9))
ax.scatter([0], [0], color="#222222", s=35)
ax.text(0.04, 0.05, "p")
ax.text(0, r_large + 0.08, "coordinate ball", ha="center", color="#2a6fbb")
ax.text(0, -r_small - 0.13, "compact closure inside chart", ha="center", color="#2f704b")
ax.set_title("Interior model: ball in R^2")
ax.set_aspect("equal")
ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-1.15, 1.15)
ax.set_xticks([])
ax.set_yticks([])

ax = axes[1]
ax.add_patch(Rectangle((-1.08, 0), 2.16, 1.1, facecolor="#f7f7f7", edgecolor="none", zorder=0))
ax.axhline(0, color="#222222", lw=2)
ax.add_patch(Wedge((0, 0), r_large, 0, 180, facecolor="#edf3fb", edgecolor="#2a6fbb", lw=2, alpha=0.8))
ax.add_patch(Wedge((0, 0), r_small, 0, 180, facecolor="#cfe8d8", edgecolor="#4c9f70", lw=2, alpha=0.95))
ax.scatter([0], [0], color="#222222", s=35, zorder=4)
ax.text(0.04, 0.04, "p on boundary")
ax.text(0, r_large + 0.08, "regular half-ball", ha="center", color="#2a6fbb")
ax.text(0, -0.13, "boundary line", ha="center")
ax.set_title("Boundary model: half-ball in H^2")
ax.set_aspect("equal")
ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-0.25, 1.15)
ax.set_xticks([])
ax.set_yticks([])
half_ball_png = save_matplotlib(fig, FIGURES / "local-compactness-half-balls.png")
plt.close(fig)

xx = np.linspace(-8, 8, 1600)
zeros = np.arange(-6, 7) * np.pi
proper_checks = {
    "local_compactness_half_ball": {
        "r_small": r_small,
        "r_large": r_large,
        "closure_contained_in_larger_half_ball": r_small < r_large,
        "closed_half_ball_chart_reason": "closed and bounded subset of R^2, hence compact by Heine-Borel",
    },
    "proper_map_x_squared": {"target_compact_interval": [0, 4], "preimage": [-2, 2], "preimage_is_closed_and_bounded": True},
    "nonproper_map_sine": {
        "target_compact_set": [0],
        "sampled_zero_count": int(len(zeros)),
        "sampled_zero_min": float(zeros.min()),
        "sampled_zero_max": float(zeros.max()),
        "preimage_unbounded_reason": "sin(x)=0 at k*pi for all integers k",
    },
}
proper_check_path = save_json(proper_checks, CHECKS / "local-compactness-properness-checks.json")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.16, subplot_titles=("Proper example: f(x)=x^2 and compact target [0,4]", "Nonproper example: g(x)=sin(x) and compact target {0}"))
fig.add_trace(go.Scatter(x=xx, y=xx**2, mode="lines", line=dict(color="#2a6fbb"), name="x^2"), row=1, col=1)
fig.add_hrect(y0=0, y1=4, fillcolor="#4c9f70", opacity=0.18, line_width=0, row=1, col=1)
fig.add_vrect(x0=-2, x1=2, fillcolor="#c64f4f", opacity=0.14, line_width=0, row=1, col=1)
fig.add_trace(go.Scatter(x=xx, y=np.sin(xx), mode="lines", line=dict(color="#5a5a5a"), name="sin(x)"), row=2, col=1)
fig.add_hline(y=0, line=dict(color="#4c9f70", width=2), row=2, col=1)
fig.add_trace(go.Scatter(x=zeros, y=np.zeros_like(zeros), mode="markers", marker=dict(color="#c64f4f", size=8), name="zeros k*pi"), row=2, col=1)
fig.update_yaxes(range=[-1, 10], row=1, col=1)
fig.update_yaxes(range=[-1.3, 1.3], row=2, col=1)
fig.update_xaxes(title_text="domain coordinate x", row=2, col=1)
fig.update_layout(height=680, width=900, template="plotly_white", margin=dict(l=45, r=30, t=85, b=50))
proper_html = save_plotly_html(fig, HTML / "proper-map-preimage-test.html")

display_artifact(book_local(half_ball_png), width=880)
display_artifact(book_local(proper_html), width=900, height=700)
display_artifact(book_local(proper_check_path))
proper_checks


## 5. Applied Lab: Classify the Space Cards

Use the chapter's implications as a checklist rather than as isolated vocabulary. Start with path witnesses: if every pair of points can be joined by a path, connectedness follows. Then ask whether the space is locally path-connected; if it is, components and path components agree. Compactness is a separate question: in Euclidean examples, closed and bounded is the decisive Heine-Borel test, while quotient and image arguments can transfer compactness to less obvious spaces. Local compactness asks for compact closures in arbitrarily small neighborhoods, which is why manifolds with boundary pass the same local test as manifolds without boundary.

The table deliberately mixes manifolds, subspaces, and classical counterexamples. The implication columns are checked by code. The entries marked as classical facts are not proved by the table; the table records how each example functions as a test case for the chapter.


In [ ]:
space_cards = [
    {"space": "closed interval [0,1]", "connected": True, "path_connected": True, "locally_connected": True, "locally_path_connected": True, "compact": True, "locally_compact": True, "manifold_boundary_model": "1-manifold with boundary", "reason_to_inspect": "endpoints have half-interval neighborhoods"},
    {"space": "R without {0}", "connected": False, "path_connected": False, "locally_connected": True, "locally_path_connected": True, "compact": False, "locally_compact": True, "manifold_boundary_model": "disjoint union of open intervals", "reason_to_inspect": "two clopen half-lines in the subspace"},
    {"space": "punctured plane R^2 without {0}", "connected": True, "path_connected": True, "locally_connected": True, "locally_path_connected": True, "compact": False, "locally_compact": True, "manifold_boundary_model": "2-manifold without boundary", "reason_to_inspect": "paths can detour around the missing point"},
    {"space": "topologist sine curve T", "connected": True, "path_connected": False, "locally_connected": False, "locally_path_connected": False, "compact": True, "locally_compact": True, "manifold_boundary_model": "not a manifold", "reason_to_inspect": "connected closure of an oscillating graph, but no path to the limit segment"},
    {"space": "rational plane Q^2", "connected": False, "path_connected": False, "locally_connected": False, "locally_path_connected": False, "compact": False, "locally_compact": False, "manifold_boundary_model": "not locally Euclidean", "reason_to_inspect": "irrational coordinate cuts separate any two-point subset"},
    {"space": "closed unit disk", "connected": True, "path_connected": True, "locally_connected": True, "locally_path_connected": True, "compact": True, "locally_compact": True, "manifold_boundary_model": "2-manifold with boundary", "reason_to_inspect": "boundary circle has half-ball neighborhoods"},
    {"space": "closed upper half-plane H^2", "connected": True, "path_connected": True, "locally_connected": True, "locally_path_connected": True, "compact": False, "locally_compact": True, "manifold_boundary_model": "2-manifold with boundary", "reason_to_inspect": "boundary line points have half-ball neighborhoods"},
]
space_df = pd.DataFrame(space_cards)
lab_table_path = save_csv(space_cards, TABLES / "space-property-lab.csv")
implication_checks = {
    "path_connected_implies_connected": bool((~space_df["path_connected"] | space_df["connected"]).all()),
    "local_path_connected_implies_local_connected": bool((~space_df["locally_path_connected"] | space_df["locally_connected"]).all()),
    "locally_path_connected_and_connected_implies_path_connected": bool((~(space_df["locally_path_connected"] & space_df["connected"]) | space_df["path_connected"]).all()),
    "compact_examples": space_df.loc[space_df["compact"], "space"].tolist(),
    "locally_compact_noncompact_examples": space_df.loc[space_df["locally_compact"] & ~space_df["compact"], "space"].tolist(),
}
lab_check_path = save_json(implication_checks, CHECKS / "space-property-lab-checks.json")
display(space_df)
display_artifact(book_local(lab_table_path))
display_artifact(book_local(lab_check_path))
implication_checks


## Proof and Invariant Scaffolds

The notebook's checks are intentionally modest, because the chapter's theorems are topological rather than numerical. Still, each check mirrors a proof move.

- The finite connectedness check verifies that locally constant binary labels are counted by `2 ** number_of_components`. This is the finite version of the clopen characterization and the theorem that maps from connected spaces to discrete spaces are constant.
- The path check verifies that the straight-line path stays inside the convex disk and records the increasingly dense oscillation of the sine curve sample. The mathematical lesson is that closure can preserve connectedness while path connectedness may fail.
- The compactness check uses exact interval merging, not sample points, to certify the finite cover of `[0,1]`. It also produces an explicit missed point for each finite initial segment of the cover `(1/n,1)`.
- The half-ball check records the radius containment that makes a smaller closure compact inside a larger chart neighborhood. This is the local compactness proof in the boundary model.
- The properness check compares a bounded preimage with an unbounded compact fiber preimage. That is the sequence-to-infinity obstruction used later in the chapter.

## Final Sanity Checks

The final `final_sanity` cell collects the JSON checks, validates core implications, asserts artifact existence and nonzero file size, and confirms that generated PNGs are nonblank.


In [ ]:
expected_artifacts = [
    CHECKS / "visual-storyboard.json",
    CHECKS / "connectedness-finite-probes.json",
    CHECKS / "proof-dependency-checks.json",
    CHECKS / "path-connectedness-checks.json",
    CHECKS / "compactness-cover-checks.json",
    CHECKS / "local-compactness-properness-checks.json",
    CHECKS / "space-property-lab-checks.json",
    TABLES / "space-property-lab.csv",
    FIGURES / "connectedness-clopen-and-components.png",
    FIGURES / "proof-dependency-map.png",
    FIGURES / "path-connectedness-sine-curve.png",
    FIGURES / "local-compactness-half-balls.png",
    HTML / "compactness-open-covers.html",
    HTML / "proper-map-preimage-test.html",
]
assert_artifacts(expected_artifacts, min_bytes=64)

storyboard = json.loads((CHECKS / "visual-storyboard.json").read_text(encoding="utf-8"))
component_data = json.loads((CHECKS / "connectedness-finite-probes.json").read_text(encoding="utf-8"))
proof_data = json.loads((CHECKS / "proof-dependency-checks.json").read_text(encoding="utf-8"))
cover_data = json.loads((CHECKS / "compactness-cover-checks.json").read_text(encoding="utf-8"))
proper_data = json.loads((CHECKS / "local-compactness-properness-checks.json").read_text(encoding="utf-8"))
lab_data = json.loads((CHECKS / "space-property-lab-checks.json").read_text(encoding="utf-8"))

assert len(storyboard) >= 6
assert component_data["connected_probe"]["component_count"] == 1
assert component_data["connected_probe"]["locally_constant_binary_label_count"] == 2
assert component_data["disconnected_probe"]["component_count"] == 2
assert component_data["disconnected_probe"]["locally_constant_binary_label_count"] == 4
assert proof_data["is_dag"] and proof_data["required_nodes_present"]
assert cover_data["closed_interval_cover"]["covers_closed_0_1"] is True
assert all(item["missed_point_is_in_open_interval"] and not item["covered_by_first_N"] for item in cover_data["finite_initial_segment_tests"])
assert proper_data["local_compactness_half_ball"]["closure_contained_in_larger_half_ball"] is True
assert proper_data["proper_map_x_squared"]["preimage_is_closed_and_bounded"] is True
assert lab_data["path_connected_implies_connected"] is True
assert lab_data["local_path_connected_implies_local_connected"] is True
assert lab_data["locally_path_connected_and_connected_implies_path_connected"] is True

png_stats = [image_stats(path) for path in expected_artifacts if path.suffix.lower() == ".png"]
assert all(stat["width"] >= 500 and stat["height"] >= 250 for stat in png_stats)
assert all(stat["max_channel_stddev"] > 3.0 for stat in png_stats)

final_sanity = {
    "source_span": "printed pp. 85-126; PDF pp. 103-144",
    "notebook": "chapter-04-connectedness-and-compactness/04-connectedness-and-compactness.ipynb",
    "artifact_count_checked": len(expected_artifacts),
    "storyboard_item_count": len(storyboard),
    "component_invariants": component_data,
    "compactness_cover_ok": cover_data["closed_interval_cover"]["covers_closed_0_1"],
    "noncompact_cover_finite_tests": cover_data["finite_initial_segment_tests"],
    "properness_checks": proper_data,
    "lab_implication_checks": lab_data,
    "png_stats": png_stats,
}
final_sanity_path = save_json(final_sanity, CHECKS / "final-sanity.json")
assert_artifacts([final_sanity_path], min_bytes=64)
display(pd.DataFrame(png_stats))
display_artifact(book_local(final_sanity_path))
final_sanity


## Takeaways

Connectedness is best read as the absence of a nontrivial continuous yes/no decision. The clopen characterization, constant-map-to-discrete-spaces result, and component partition are all versions of that same idea.

Path connectedness is stronger in arbitrary spaces, but manifolds are locally path-connected, so for manifolds connectedness and path connectedness agree. This is why path construction is such a useful proof strategy in manifold topology.

Compactness converts infinite local data into finite certificates. It is preserved by continuous images, closed subsets of compact spaces stay compact, compact subsets of Hausdorff spaces are closed, and compact Hausdorff domains make many quotient and embedding arguments cleaner through the closed-map lemma.

Local compactness is the compactness property that survives at small scale. Manifolds with boundary pass the test because half-ball charts have smaller neighborhoods with compact closures, just as full Euclidean balls do.

Proper maps extend compactness from spaces to maps: compact target sets must have compact preimages. In first countable Hausdorff settings, this is visible through sequences escaping to infinity.
